BIBLIOTECAS

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import joblib

from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.discriminant_analysis import StandardScaler
from sklearn.pipeline import Pipeline

from lime.lime_tabular import LimeTabularExplainer
from IPython.display import HTML

import shap
shap.initjs();
from shap import Explainer
from shap.plots import bar
from shap.plots import waterfall
from shap.plots import force
from shap.plots import beeswarm

LEITURA DO DATASET FINAL

In [ ]:
df_ind = pd.read_csv('df_ind_distancia.csv', sep=';', encoding='utf-8-sig')
X = df_ind.drop(columns=['desc_cod_etnia_1_novacod'])
y = df_ind['desc_cod_etnia_1_novacod']

DICIONÁRIO

In [ ]:
dicionario_variaveis = {
    'VAL_LATITUDE': 'Latitude',
    'VAL_LONGITUDE': 'Longitude',
    'cod_setor': 'Setor censitário',
    'distancia': 'Distância de Damerau-Levenshtein entrada-descritor',
    'txt_lingua_entrada_coleta': 'Texto de entrada da língua',
    'txt_etnia_entrada_coleta': 'Texto de entrada da etnia',
    'desc_cod_etnia_1_novacod': 'Etnia após codificação',
    'desc_cod_lingua_1_novacod': 'Língua após codificação',
    'tipo_setor': 'Código do tipo do setor',
    'CD_TI': 'Código da terra indígena',
    'sobrenome': 'Sobrenome',
    'descricao_mais_proxima': 'Descritor mais similar por Damerau-Levenshtein',
    'Predicao': 'Língua predição',
    'Real': 'Língua após codificação',
    'Confianca_Predita': 'Confiança'
}


def formata_colunas(tabela):
    tabela = tabela.rename(columns=dicionario_variaveis)
    return tabela

def formata_indexs(tabela):
    tabela = tabela.rename(index=dicionario_variaveis)
    return tabela

dicionario_proc = {
    'num__VAL_LATITUDE': 'Latitude',
    'num__VAL_LONGITUDE': 'Longitude',
    'num__cod_setor': 'Setor censitário',
    'num__distancia': 'Distância de Damerau-Levenshtein entrada-descritor',
    'cat__txt_lingua_entrada_coleta': 'Texto de entrada da língua',
    'cat__txt_etnia_entrada_coleta': 'Texto de entrada da etnia',
    'cat__desc_cod_etnia_1_novacod': 'Etnia após codificação',
    'cat__desc_cod_lingua_1_novacod': 'Língua após codificação',
    'cat__tipo_setor': 'Código do tipo do setor',
    'cat__CD_TI': 'Código da terra indígena',
    'cat__sobrenome': 'Sobrenome',
    'cat__descricao_mais_proxima': 'Descritor mais similar por Damerau-Levenshtein',
    'Predicao': 'Etnia predição',
    'Real': 'Etnia após codificação',
    'Confianca_Predita': 'Confiança'
}


def formata_proc(tabela, coluna):
    tabela[coluna] = tabela[coluna].replace(dicionario_proc)
    return tabela

def formata_array(vetor):
    substituir = np.vectorize(lambda x: dicionario_proc.get(x, x))
    return substituir(vetor)

CARREGAMENTO DO MODELO VENCEDOR

In [ ]:
modelo = joblib.load('floresta_parametros.pkl')

PREDIÇÕES TOTAIS DO MODELO VENCEDOR

In [ ]:
y_pred = modelo.predict(X)

EXIBIÇÃO DAS PROBABILIDADES DAS CATEGORIAS DOS RÓTULOS PREDITOS (CATEGORIA DE MAIOR PROBABILIDADE POR PREDIÇÃO) - TOTAL

In [ ]:
probas = modelo.predict_proba(X)  

confianca_predita = np.max(probas, axis=1) 

resultados = pd.DataFrame({
    'Registro': range(len(X)),
    'Predicao': y_pred,
    'Real': y,
    'Confianca_Predita': confianca_predita
})

final = X.join(resultados)
formata_colunas(final)

GRAVAÇÃO DAS PREDIÇÕES - TOTAL

In [ ]:
formata_colunas(final).to_csv('df_ind_predicoes_tota_parametros.csv', index=False, sep=';', encoding='utf-8-sig')

ERROS DE PREDIÇÃO

In [ ]:
erros = final[~(
     (final['Predicao'] == final['Real']))
].sort_values('Confianca_Predita', ascending=False)

formata_colunas(erros)

GRAVAÇÃO DAS PREDIÇÕES - ERROS

In [ ]:
formata_colunas(erros).to_csv('df_ind_predicoes_erros_parametros.csv', index=False, sep=';', encoding='utf-8-sig')

ERROS DE PREDIÇÃO ONDE O RÓTULO REAL É IGUAL A 'NÃO INFORMADO'

In [ ]:
formata_colunas(erros[erros['Real'].isin(['NÃO INFORMADO'])].sort_values('Confianca_Predita', ascending=False))

GRAVAÇÃO DAS PREDIÇÕES - ERROS DE PREDIÇÃO ONDE O RÓTULO REAL É IGUAL A 'NÃO INFORMADO'

In [ ]:
formata_colunas(erros[erros['Real'].isin(['NÃO INFORMADO'])].sort_values('Confianca_Predita', ascending=False)).to_csv('df_ind_predicoes_erros_real_ni_parametros.csv', index=False, sep=';', encoding='utf-8-sig')

ERROS DE PREDIÇÃO ONDE O RÓTULO PREDITO É IGUAL A 'NÃO INFORMADO'

In [ ]:
formata_colunas(erros[erros['Predicao'].isin(['NÃO INFORMADO'])].sort_values('Confianca_Predita', ascending=False))

GRAVAÇÃO DAS PREDIÇÕES - ERROS DE PREDIÇÃO ONDE O RÓTULO REAL É IGUAL A 'NÃO INFORMADO'

In [ ]:
formata_colunas(erros[erros['Predicao'].isin(['NÃO INFORMADO'])].sort_values('Confianca_Predita', ascending=False)).to_csv('df_ind_predicoes_erros_predito_ni_parametros.csv', index=False, sep=';', encoding='utf-8-sig')

ERROS DE PREDIÇÃO DESCONSIDERANDO CASOS ONDE O RÓTULO REAL OU O RÓTULO PREDITO SÃO 'NÃO INFORMADO')

In [ ]:
formata_colunas(erros[(erros['Predicao']!='NÃO INFORMADO')&(erros['Real']!='NÃO INFORMADO')].sort_values('Confianca_Predita', ascending=False))

GRAVAÇÃO DAS PREDIÇÕES - ERROS DE PREDIÇÃO DESCONSIDERANDO CASOS ONDE O RÓTULO REAL OU O RÓTULO PREDITO SÃO 'NÃO INFORMADO')

In [ ]:
formata_colunas(erros[(erros['Predicao']!='NÃO INFORMADO')&(erros['Real']!='NÃO INFORMADO')].sort_values('Confianca_Predita', ascending=False)).to_csv('df_ind_predicoes_erros_sem_ni_parametros.csv', index=False, sep=';', encoding='utf-8-sig')

QUANTIDADES DE ERROS DE PREDIÇÃO DESCONSIDERANDO CASOS ONDE O RÓTULO REAL OU O RÓTULO PREDITO SÃO 'NÃO INFORMADO') POR LÍNGUA DE ENTRADA, DESCRIÇÃO MAIS PRÓXIMA, VALORES PREDITO E REAL

In [ ]:
formata_colunas(erros[
        (erros['Predicao'] != 'NÃO INFORMADO') &
        (erros['Real'] != 'NÃO INFORMADO')
    ].value_counts([
        'txt_etnia_entrada_coleta',
        'descricao_mais_proxima',
        'Predicao',
        'Real'
    ]).reset_index(name='frequencia'))


GRAVAÇÃO - QUANTIDADES DE ERROS DE PREDIÇÃO DESCONSIDERANDO CASOS ONDE O RÓTULO REAL OU O RÓTULO PREDITO SÃO 'NÃO INFORMADO') POR LÍNGUA DE ENTRADA, DESCRIÇÃO MAIS PRÓXIMA, VALORES PREDITO E REAL

In [ ]:
formata_colunas(erros[
        (erros['Predicao'] != 'NÃO INFORMADO') &
        (erros['Real'] != 'NÃO INFORMADO')
    ].value_counts([
        'txt_etnia_entrada_coleta',
        'descricao_mais_proxima',
        'Predicao',
        'Real'
    ]).reset_index(name='frequencia')).to_csv('df_ind_qtds_agrupado_parametros.csv', index=False, sep=';', encoding='utf-8-sig')

IMPORTÂNCIA DAS VARIÁVEIS

In [ ]:
df_importancia = pd.DataFrame({"Variáveis":
                               modelo["preprocessor"].get_feature_names_out(),
                               "Importância":
                               modelo["classifier"].feature_importances_}).sort_values('Importância', ascending=False)

formata_proc(df_importancia,'Variáveis')

In [ ]:
fig = plt.figure(figsize=(10, 4))
plt.xticks(rotation=90)
sns.barplot(data=formata_proc(df_importancia,'Variáveis'), x="Variáveis", y="Importância");

PIPELINES DE PRÉ-PROCESSAMENTO COM ORDINAL E ONE HOT ENCONDERS

In [ ]:
variaveis_numericas = ['VAL_LATITUDE','VAL_LONGITUDE','cod_setor','distancia']
variaveis_categoricas= ['txt_etnia_entrada_coleta','desc_cod_lingua_1_novacod','tipo_setor', 'CD_TI', 'sobrenome','descricao_mais_proxima']

numeric_pipeline = Pipeline(steps=[
    ('scaler', StandardScaler())
])

categorical_pipeline_ordinal = Pipeline(steps=[
    ('encoder', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))
])

categorical_pipeline_onehot = Pipeline(steps=[
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor_ordinal = ColumnTransformer(
    transformers=[
        ('num', numeric_pipeline, variaveis_numericas),
        ('cat', categorical_pipeline_ordinal, variaveis_categoricas)
    ]
)

preprocessor_onehot = ColumnTransformer(
    transformers=[
        ('num', numeric_pipeline, variaveis_numericas),
        ('cat', categorical_pipeline_onehot, variaveis_categoricas)
    ]
)

display(preprocessor_ordinal)
display(preprocessor_onehot)

EXPLICABILIDADE LIME

In [ ]:
#  Transformação do dataset no formato pré-processado do modelo
X_trans = preprocessor_ordinal.fit_transform(X)

# Criação do Explainer Lime

explainer = LimeTabularExplainer(
    training_data=X_trans,
    feature_names=formata_array(preprocessor_ordinal.get_feature_names_out()),
    class_names=list(modelo["classifier"].classes_),
    mode="classification"
)

EXPLICABILIDADE LIME PARA UMA INSTÂNCIA

In [ ]:
# instância aleatória
item = 5585

exp = explainer.explain_instance(
    X_trans[item], 
    modelo["classifier"].predict_proba,
    top_labels=1
)

classe_vencedora = exp.available_labels()[0]

display(formata_colunas(final.iloc[[item]]))

html_str = exp.as_html(labels=(classe_vencedora,))

# Aumenta a largura do contêiner principal do LIME via substituição de string
html_str = html_str.replace(".lime {", ".lime { width: 500px !important;")

display(HTML(html_str))

fig = exp.as_pyplot_figure(label=classe_vencedora)

# Ajusta o tamanho da figura (Largura, Altura) em polegadas
fig.set_size_inches(3, 4) 
plt.tight_layout()
plt.show()

EXPLICABILIDADE SHAP

In [ ]:
X_item = X_trans[item : item + 1]

explicador = shap.Explainer(modelo["classifier"].predict_proba, X_trans)

shap_values_item = explicador(X_item)

feature_names = list(formata_array(preprocessor_ordinal.get_feature_names_out()))
shap_values_item.feature_names = feature_names

probs = modelo["classifier"].predict_proba(X_item)[0]
classe_vencedora = np.argmax(probs)

nome_classe_vencedora = modelo["classifier"].classes_[classe_vencedora]

display(formata_colunas(final.iloc[[item]]))
print(f"Explicando a Classe: {nome_classe_vencedora} (Probabilidade: {probs[classe_vencedora]:.2f})")
shap.plots.waterfall(shap_values_item[0, :, classe_vencedora])
shap.plots.force(shap_values_item[0, :, classe_vencedora])

SHAP GLOBAL COM AMOSTRA DE 1000 PARA A CLASSE VENCEDORA

In [ ]:
X_sample = shap.utils.sample(X_trans, 1000)
shap_values_global = explicador(X_sample)

shap_values_global.feature_names = feature_names

print(f"Explicabilidade Global para a Classe: {nome_classe_vencedora}")
shap.plots.bar(shap_values_global[:, :, classe_vencedora])
shap.plots.beeswarm(shap_values_global[:, :, classe_vencedora])